# Symmetry-Adapted Density Fitting: ERI Blocks

Compute MO electron repulsion integrals (ERIs) using either **FFTDF** (FFT density fitting) or
**GDF** (Gaussian density fitting) in two complementary representations:

| Representation | Shape | Description |
|---|---|---|
| **k-resolved 7D** | `(nk, nk, nk, n1, n2, n3, n4)` | complex, full k-point resolved |
| **Symmetry-adapted** | `(n1, n2, n3, n4)` | real, k-folded supercell form |

Four MO ERI blocks are covered:

| Block | Indices |
|---|---|
| **(oo\|oo)** | occupied-occupied \| occupied-occupied |
| **(vv\|vv)** | virtual-virtual \| virtual-virtual |
| **(ov\|vo)** | occupied-virtual \| virtual-occupied |
| **(oo\|vv)** | occupied-occupied \| virtual-virtual |

The symmetry-adapted form sums the 7D tensor over all k-point triplets,
exploiting translational symmetry (momentum conservation) and time-reversal symmetry
(result is real):
$$\mathrm{eri\_spc}[p,q,r,s] = \sum_{k_1,k_2,k_3} \mathrm{eri\_7d}[k_1,k_2,k_3,p,q,r,s]$$


In [1]:
import sys
from pathlib import Path
import pickle
import time

import numpy as np
from pyscf.pbc import df
from pyscf.pbc.lib import kpts_helper

repo_root = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
sys.path.insert(0, str(repo_root))

import fft
import utils




## Configuration

In [2]:
nk           = 4         # k-mesh = nk x nk x nk  (2 or 3)
df_type      = "GDF"      # "GDF" or "FFTDF"
basis        = "gth-dzvp"
ke_cutoff    = 40.0
max_mem_gb   = 8.0        # PySCF DF memory hint, in GB

# The GTH pseudopotential is the frozen-core treatment here. For diamond/gth-dzvp
# this leaves 26 active MOs per primitive cell; keep all of them in the active space.
frozen_core = True
nmo_per_cell = 26

kmesh  = np.array([nk, nk, nk])
klabel = f"{kmesh[0]}x{kmesh[1]}x{kmesh[2]}"

save_npz = False
out_npz = repo_root / "data" / f"ERI_{df_type.upper()}_blocks_diamond_{klabel}_{basis}_ke{ke_cutoff}.npz"

# ERI construction controls. The first ERI cell no longer builds every block;
# it builds only this union by default. Later cells call get_eri_7d(block), so
# adding a block below computes it on demand.
spc_blocks = ["oooo", "ovvo"]
run_symmetry_checks = False
symmetry_check_blocks = ["oooo", "ovvo"]

# Norm computation controls. Quantity 1 needs SVD factors and is the expensive part.
# Use randomized/truncated SVD by default and skip (vv|vv); set q1_svd_method="full"
# and include "vvvv" only when you explicitly need exact slow numbers.
q1_blocks = ["oooo", "ovvo"]
q1_svd_method = "randomized"   # "randomized", "svds", or "full"
q1_max_rank = 96                # ignored by "full"
q1_oversample = 16
q1_niter = 2
q1_tol = 1e-8
q1_random_seed = 1234

# (vv|vv) is costly to build for GDF. Add "vvvv" here only when you need it.
norm2_blocks = ["oooo", "ovvo"]
norm2_tol = 1e-6

eri_blocks_to_build = sorted(set(spc_blocks) | set(q1_blocks) | set(norm2_blocks))
use_cached_gdf_ao2mo = True  # reuse GDF MO-pair tensors across k-triplets
cache_eri_7d = True
overwrite_eri_cache = False
load_eri_cache_mmap = False
eri_cache_dir = repo_root / "data" / "eri_7d_cache"




## Load SCF and Density Fitting Object

Load the pre-computed FFTDF SCF pickle from `data/`, then attach the requested density fitting backend.

For `df_type = "GDF"`, the notebook intentionally mirrors `do_scf_gdf.py`: construct `df.GDF(cell, kpts)`, use `data_GDF/GDF_diamond_...chk`, load the cache if present, otherwise build and save it, and then assign `mf.with_df = make_gdf()`.

The core electrons are already frozen by the GTH pseudopotential, so the active MO space keeps all 26 orbitals per cell.


In [3]:
def paths_for_mesh(klabel):
    scf_pkl = repo_root / "data" / f"SCF_diamond_{klabel}_{basis}_ke{ke_cutoff}.pkl"
    scf_gdf_pkl = repo_root / "data_GDF" / f"SCF_diamond_{klabel}_{basis}_ke{ke_cutoff}.pkl"
    gdf_chk = repo_root / "data_GDF" / f"GDF_diamond_{klabel}_{basis}_ke{ke_cutoff}.chk"
    return scf_pkl, scf_gdf_pkl, gdf_chk


def load_scf():
    scf_pkl, scf_gdf_pkl, gdf_chk = paths_for_mesh(klabel)
    scf_candidates = [scf_pkl, scf_gdf_pkl]
    scf_to_load = next((path for path in scf_candidates if path.exists()), scf_pkl)
    if not scf_to_load.exists():
        msg = " or ".join(str(path) for path in scf_candidates)
        raise FileNotFoundError(
            f"Missing SCF pickle for {klabel}: {msg}. "
            f"Generate it with `python do_scf_fft.py {nk}`."
        )

    with open(scf_to_load, "rb") as f:
        mf = pickle.load(f)
    return mf, scf_to_load, gdf_chk


def make_gdf(cell, kpts, gdf_chk):
    """Build/load Gaussian DF exactly like do_scf_gdf.py."""
    gdf_chk.parent.mkdir(parents=True, exist_ok=True)
    with_df = df.GDF(cell, kpts)
    with_df.verbose = 0
    with_df.max_memory = int(max_mem_gb * 1000)
    if gdf_chk.exists():
        print("Loading GDF cache =", gdf_chk, flush=True)
        with_df._cderi = str(gdf_chk)
    else:
        print("Building GDF cache =", gdf_chk, flush=True)
        with_df._cderi_to_save = str(gdf_chk)
        with_df.build()
        with_df._cderi = str(gdf_chk)
        with_df._cderi_to_save = None
    return with_df


def make_fftdf(cell, kpts, mf):
    if isinstance(mf.with_df, df.FFTDF):
        with_df = mf.with_df
    else:
        with_df = df.FFTDF(cell, kpts)
        with_df.verbose = 0
        with_df.build()
    with_df.max_memory = int(max_mem_gb * 1000)
    return with_df


mf, scf_pkl, gdf_chk = load_scf()
cell = mf.cell
kpts = cell.make_kpts(kmesh)
kpts_int = np.round(cell.get_scaled_kpts(kpts) * kmesh).astype(int) % kmesh
assert utils.is_k_ordered(kpts_int, kmesh)
kconserv3 = kpts_helper.get_kconserv(cell, kpts)

if df_type.upper() == "GDF":
    print("GDF chkfile =", gdf_chk, flush=True)
    with_df = make_gdf(cell, kpts, gdf_chk)
elif df_type.upper() == "FFTDF":
    with_df = make_fftdf(cell, kpts, mf)
else:
    raise ValueError(f"Unknown df_type={df_type!r}; expected 'FFTDF' or 'GDF'.")
mf.with_df = with_df

C               = np.asarray(mf.mo_coeff)   # (nkpts, nao, nmo)
nkpts, nao, nmo = C.shape
if frozen_core and nmo != nmo_per_cell:
    raise ValueError(f"Expected {nmo_per_cell} frozen-core active MOs per cell, got {nmo}.")

ncore = 0                     # GTH pseudopotential has already frozen the core.
nocc  = cell.nelectron // 2   # active occupied orbitals per k-point
nvir  = nmo - nocc

Cact = np.array(C[:, :, ncore:nmo_per_cell], order="C", copy=True)
Cocc = np.array(C[:, :, ncore:nocc], order="C", copy=True)
Cvir = np.array(C[:, :, nocc:nmo_per_cell], order="C", copy=True)

print(f"SCF loaded : {scf_pkl}")
print(f"DF request : {df_type.upper()}")
print(f"DF object  : {type(with_df).__name__}")
if df_type.upper() == "GDF":
    print(f"GDF cache  : {gdf_chk}")
print(f"frozen core: {frozen_core}, ncore = {ncore}, active MOs/cell = {Cact.shape[2]}")
print(f"cell.mesh  = {cell.mesh}")
print(f"kmesh      = {kmesh},  nkpts = {nkpts}")
print(f"nao = {nao},  nocc = {nocc},  nvir = {nvir},  nmo = {nmo}")



GDF chkfile = /mnt/c/Users/Jielun/Documents/GitHub/Jiace/fftisdf/data_GDF/GDF_diamond_4x4x4_gth-dzvp_ke40.0.chk
Loading GDF cache = /mnt/c/Users/Jielun/Documents/GitHub/Jiace/fftisdf/data_GDF/GDF_diamond_4x4x4_gth-dzvp_ke40.0.chk
SCF loaded : /mnt/c/Users/Jielun/Documents/GitHub/Jiace/fftisdf/data/SCF_diamond_4x4x4_gth-dzvp_ke40.0.pkl
DF request : GDF
DF object  : GDF
GDF cache  : /mnt/c/Users/Jielun/Documents/GitHub/Jiace/fftisdf/data_GDF/GDF_diamond_4x4x4_gth-dzvp_ke40.0.chk
frozen core: True, ncore = 0, active MOs/cell = 26
cell.mesh  = [15, 15, 15]
kmesh      = [4 4 4],  nkpts = 64
nao = 26,  nocc = 4,  nvir = 22,  nmo = 26


## ERI Block Definitions

Each block is characterised by four frozen-core active-space MO coefficient arrays `[C1, C2, C3, C4]` corresponding to indices in `(p q | r s)`.

With the GTH pseudopotential there are no explicit core MOs to drop in the loaded SCF object: `ncore = 0`, `nocc = 4`, `nvir = 22`, and `nmo = 26` per cell.


In [4]:
block_config = {
    "oooo": {"coeff": [Cocc, Cocc, Cocc, Cocc], "label": "(oo|oo)"},
    "vvvv": {"coeff": [Cvir, Cvir, Cvir, Cvir], "label": "(vv|vv)"},
    "ovvo": {"coeff": [Cocc, Cvir, Cvir, Cocc], "label": "(ov|vo)"},
    "oovv": {"coeff": [Cocc, Cocc, Cvir, Cvir], "label": "(oo|vv)"},
}

## k-Resolved 7D ERI Blocks

`ao2mo_7d_for_df` evaluates
$$( p^{k_1} q^{k_2} | r^{k_3} s^{k_4} ) =
  \frac{1}{N_k}\sum_G \tilde\rho_{pq}^*(G,\mathbf{q})\,v(G+\mathbf{q})\,\tilde\rho_{rs}(G,\mathbf{q}),
  \quad \mathbf{q} = k_1 - k_2$$
returning a complex array of shape `(nkpts, nkpts, nkpts, n1, n2, n3, n4)`.
$k_4$ is fixed by momentum conservation $k_1 - k_2 = k_3 - k_4$.

For FFTDF/FFTISDF objects with a native `ao2mo_7d`, the helper uses that implementation. For GDF objects, it loops over k-point triplets and calls PySCF's Gaussian-DF `ao2mo`.


In [5]:
def _format_mo_coeff_kpts(mo_coeff_kpts):
    if isinstance(mo_coeff_kpts, np.ndarray) and mo_coeff_kpts.ndim == 3:
        mo_coeff_kpts = [mo_coeff_kpts] * 4
    else:
        mo_coeff_kpts = list(mo_coeff_kpts)
    if len(mo_coeff_kpts) != 4:
        raise ValueError("Expected four MO coefficient arrays.")
    return [np.asarray(c) for c in mo_coeff_kpts]


def ao2mo_7d_for_df(with_df, mo_coeff_kpts, kpts):
    """Return a 7D k-resolved MO ERI tensor for FFTDF-like or GDF objects."""
    native_ao2mo_7d = getattr(with_df, "ao2mo_7d", None)
    if callable(native_ao2mo_7d):
        return native_ao2mo_7d(mo_coeff_kpts, kpts=kpts)

    mo_coeff_kpts = _format_mo_coeff_kpts(mo_coeff_kpts)
    dims = tuple(c.shape[2] for c in mo_coeff_kpts)
    out = np.empty((nkpts, nkpts, nkpts) + dims, dtype=np.complex128)

    local_kconserv3 = getattr(with_df, "kconserv3", kconserv3)

    for k1 in range(nkpts):
        for k2 in range(nkpts):
            for k3 in range(nkpts):
                k4 = int(local_kconserv3[k1, k2, k3])
                coeff = [
                    mo_coeff_kpts[0][k1],
                    mo_coeff_kpts[1][k2],
                    mo_coeff_kpts[2][k3],
                    mo_coeff_kpts[3][k4],
                ]
                kpts4 = [kpts[k1], kpts[k2], kpts[k3], kpts[k4]]
                eri = with_df.ao2mo(coeff, kpts=kpts4, compact=False)
                out[k1, k2, k3] = np.asarray(eri).reshape(dims)
    return out


_gdf_pair_tensor_cache = {}


def _as_complex_lpq(lpq_r, lpq_i):
    return np.asarray(lpq_r) + 1j * np.asarray(lpq_i)


def gdf_pair_tensor(with_df, mo_left, mo_right, kleft, kright, tag):
    """Build/cache one GDF three-index MO-pair tensor L_Q,pq."""
    key = (tag, int(kleft), int(kright))
    if key in _gdf_pair_tensor_cache:
        return _gdf_pair_tensor_cache[key]

    tensors = []
    signs = []
    kpts_pair = [kpts[kleft], kpts[kright]]
    max_memory = int(max_mem_gb * 1000)

    for lpq_r, lpq_i, sign in with_df.sr_loop(kpts_pair, max_memory=max_memory, compact=False):
        lpq = _as_complex_lpq(lpq_r, lpq_i).reshape(-1, nao, nao)
        lmo = np.einsum("xp,Qxy,yq->Qpq", mo_left[kleft].conj(), lpq, mo_right[kright], optimize=True)
        tensors.append(np.array(lmo, order="C", copy=True))
        signs.append(np.full(lmo.shape[0], sign, dtype=np.float64))

    out = (np.concatenate(tensors, axis=0), np.concatenate(signs, axis=0))
    _gdf_pair_tensor_cache[key] = out
    return out


def contract_gdf_pair_tensors(l12, sign12, l34, sign34):
    if l12.shape[0] != l34.shape[0]:
        raise ValueError(f"Auxiliary dimensions differ: {l12.shape[0]} vs {l34.shape[0]}")
    if not np.allclose(sign12, sign34):
        raise ValueError("The two pair tensors came from different signed auxiliary metrics.")
    return np.einsum("Q,Qpq,Qrs->pqrs", sign12, l12, l34.conj(), optimize=True)


def gdf_ao2mo_7d_block(with_df, bname, kpts):
    """Build one 7D ERI block from cached GDF MO-pair tensors.

    This is usually faster than repeated per-triplet ao2mo because each k-pair
    transform is reused across many k3 values.
    """
    mo1, mo2, mo3, mo4 = block_config[bname]["coeff"]
    dims = (mo1.shape[2], mo2.shape[2], mo3.shape[2], mo4.shape[2])
    out = np.empty((nkpts, nkpts, nkpts) + dims, dtype=np.complex128)
    local_kconserv3 = getattr(with_df, "kconserv3", kconserv3)

    for k1 in range(nkpts):
        for k2 in range(nkpts):
            l12, sign12 = gdf_pair_tensor(with_df, mo1, mo2, k1, k2, (bname, "12"))
            for k3 in range(nkpts):
                k4 = int(local_kconserv3[k1, k2, k3])
                l34, sign34 = gdf_pair_tensor(with_df, mo3, mo4, k3, k4, (bname, "34"))
                out[k1, k2, k3] = contract_gdf_pair_tensors(l12, sign12, l34, sign34)
    return out



In [6]:
def eri_cache_path(bname):
    return eri_cache_dir / f"{df_type.upper()}_{klabel}_{basis}_ke{ke_cutoff}_{bname}_7d.npy"


def get_eri_7d(bname):
    """Build or load one k-resolved 7D ERI block."""
    if bname in eri_7d:
        return eri_7d[bname]
    if bname not in block_config:
        raise KeyError(f"Unknown ERI block {bname!r}; choose from {list(block_config)}")

    cache_path = eri_cache_path(bname)
    if cache_eri_7d and cache_path.exists() and not overwrite_eri_cache:
        print(f"Loading {block_config[bname]['label']} from {cache_path}", flush=True)
        e7 = np.load(cache_path, mmap_mode="r" if load_eri_cache_mmap else None)
    else:
        t0 = time.perf_counter()
        cfg = block_config[bname]
        print(f"Building {cfg['label']} with {type(with_df).__name__} ...", flush=True)
        if use_cached_gdf_ao2mo and isinstance(with_df, df.GDF):
            e7 = gdf_ao2mo_7d_block(with_df, bname, kpts)
        else:
            e7 = ao2mo_7d_for_df(with_df, cfg["coeff"], kpts)
        elapsed = time.perf_counter() - t0
        print(f"  built in {elapsed:.2f} s", flush=True)
        if cache_eri_7d:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, np.asarray(e7))
            print(f"  cached to {cache_path}", flush=True)

    eri_7d[bname] = e7
    return e7


print(f"{'Block':8s}  {'Shape':44s}  {'||ERI||_F':>14s}")
print("-" * 74)

if not eri_blocks_to_build:
    print("No ERI blocks requested. Add block names to spc_blocks, q1_blocks, or norm2_blocks.")

eri_7d = {}
for bname in eri_blocks_to_build:
    cfg = block_config[bname]
    e7 = get_eri_7d(bname)
    print(f"{cfg['label']:8s}  {str(e7.shape):44s}  {np.linalg.norm(e7):14.6e}")

skipped_blocks = [b for b in block_config if b not in eri_7d]
if skipped_blocks:
    print(f"Skipped initial build for blocks: {skipped_blocks}")
    print("They can still be computed later with get_eri_7d(block_name).")



Block     Shape                                              ||ERI||_F
--------------------------------------------------------------------------
Building (oo|oo) with GDF ...


OSError: Unable to synchronously open file (bad object header version number)

## Symmetry-Adapted (Supercell) ERI Blocks

Summing the 7D tensor over all k-point triplets folds the translational symmetry:
$$\mathrm{eri\_spc}[p,q,r,s] = \sum_{k_1,k_2,k_3} \mathrm{eri\_7d}[k_1,k_2,k_3,p,q,r,s]$$

The result is **real** (time-reversal symmetry) and has shape `(n1, n2, n3, n4)` — a compact
supercell ERI that is directly usable in post-HF methods.

In [ ]:
print(f"{'Block':8s}  {'Shape':22s}  {'||ERI||_F':>14s}  {'dtype':>10s}")
print("-" * 62)

eri_spc = {}
for bname in spc_blocks:
    cfg = block_config[bname]
    es = get_eri_7d(bname).sum(axis=(0, 1, 2)).real
    eri_spc[bname] = es
    print(f"{cfg['label']:8s}  {str(es.shape):22s}  {np.linalg.norm(es):14.6e}  {str(es.dtype):>10s}")

if not spc_blocks:
    print("No symmetry-adapted blocks requested; set spc_blocks to build them.")


Block     Shape                        ||ERI||_F       dtype
--------------------------------------------------------------
(oo|oo)   (4, 4, 4, 4)              1.766136e+03     float64
(ov|vo)   (4, 22, 22, 4)            3.000567e+02     float64


## Permutation Symmetries

The exact ERIs satisfy bra–ket symmetry: `(pq|rs) = (rs|pq)`.

In the symmetry-adapted basis this means `eri_spc[p,q,r,s] = eri_spc_swap[r,s,p,q]`, where
`eri_spc_swap` is the ERI with bra and ket exchanged.

- **(oo\|oo)** and **(vv\|vv)**: bra and ket span the same space → check `es.transpose(2,3,0,1) == es` directly.
- **(ov\|vo)** and **(oo\|vv)**: the swapped block has different orbital character `(vo\|ov)` / `(vv\|oo)` → compute it and compare.

In [ ]:
def spc_eri(coeff):
    return ao2mo_7d_for_df(with_df, coeff, kpts).sum(axis=(0, 1, 2)).real

if not run_symmetry_checks:
    print("Symmetry checks skipped. Set run_symmetry_checks = True to compute swapped blocks.")
else:
    print("Bra-ket symmetry: ||(pq|rs) - (rs|pq)|| / ||(pq|rs)||")
    print("-" * 52)

    for bname in ["oooo", "vvvv"]:
        if bname not in symmetry_check_blocks:
            continue
        es = eri_spc.get(bname)
        if es is None:
            es = get_eri_7d(bname).sum(axis=(0, 1, 2)).real
            eri_spc[bname] = es
        diff = np.linalg.norm(es - es.transpose(2, 3, 0, 1))
        ref  = max(np.linalg.norm(es), 1e-300)
        print(f"  {block_config[bname]['label']:8s}  {diff / ref:.3e}  (self)")

    if "ovvo" in symmetry_check_blocks:
        es = eri_spc.get("ovvo")
        if es is None:
            es = get_eri_7d("ovvo").sum(axis=(0, 1, 2)).real
            eri_spc["ovvo"] = es
        es_voov = spc_eri([Cvir, Cocc, Cocc, Cvir])
        diff = np.linalg.norm(es - es_voov.transpose(2, 3, 0, 1))
        ref  = max(np.linalg.norm(es), 1e-300)
        print(f"  (ov|vo)   {diff / ref:.3e}  (vs (vo|ov))")

    if "oovv" in symmetry_check_blocks:
        es = eri_spc.get("oovv")
        if es is None:
            es = get_eri_7d("oovv").sum(axis=(0, 1, 2)).real
            eri_spc["oovv"] = es
        es_vvoo = spc_eri([Cvir, Cvir, Cocc, Cocc])
        diff = np.linalg.norm(es - es_vvoo.transpose(2, 3, 0, 1))
        ref  = max(np.linalg.norm(es), 1e-300)
        print(f"  (oo|vv)   {diff / ref:.3e}  (vs (vv|oo))")


Symmetry checks skipped. Set run_symmetry_checks = True to compute swapped blocks.


## k-Point Norm Statistics (7D)

How does the ERI magnitude vary across k-point triplets $(k_1, k_2, k_3)$?

In [ ]:
print(f"{'Block':8s}  {'max':>14s}  {'mean':>14s}  {'min':>14s}")
print("-" * 58)
for bname in eri_7d:
    cfg = block_config[bname]
    e7 = get_eri_7d(bname)
    norms = np.linalg.norm(e7.reshape(nkpts, nkpts, nkpts, -1), axis=-1)
    print(f"{cfg['label']:8s}  {norms.max():14.4e}  {norms.mean():14.4e}  {norms.min():14.4e}")


Block                max            mean             min
----------------------------------------------------------
(oo|oo)       2.2098e+00      1.2720e+00      3.1537e-01
(ov|vo)       7.7591e-01      6.7076e-01      5.8666e-01


## DF Factor Operator Norms via Q-Sector SVD

For each momentum transfer $Q \in \{0,\ldots,N_k{-}1\}$, define the ERI matrix
$$M^Q_{(k_1,p,q),\,(k_3,r,s)} = \mathrm{eri\_7d}[k_1,\;k_1{-}Q,\;k_3,\;p,q,r,s]$$
with shape $(N_k n_1 n_2)\times(N_k n_3 n_4)$.  The DF factorisation follows from the SVD
$M^Q = U\Sigma V^\dagger$, giving
$$L^Q = U\Sigma^{1/2}\in\mathbb{C}^{N_k n_1 n_2\times r}, \qquad
  R^Q = V\Sigma^{1/2}\in\mathbb{C}^{N_k n_3 n_4\times r}.$$

For fixed $Q$ and auxiliary index $n$, the $n$-th column of $L^Q$ reshaped as $(N_k,n_1,n_2)$
defines a **block-semi-diagonal** matrix $[L^{Q,n}]_{(k_p,p),(k_q,q)}$ (nonzero only when
$k_q = k_p - Q$), whose operator norm equals the maximum block spectral norm:
$$\|L^{Q,n}_{(k_p,p)\leftrightarrow(k_q,q)}\|_2 = \max_{k_p}\,\sigma_{\max}(L^{Q,n}[k_p,:,:])$$

**Quantity 1** — sum of block-norm products over all $(Q,n)$, computed for **(oo|oo)**, **(vv|vv)**, **(ov|vo)**:
$$\mathcal{N}_1 = \sum_{Q,n}\|L^{Q,n}_{(k_p,p)\leftrightarrow(k_q,q)}\|_2\;\|R^{Q,n}_{(k_r,r)\leftrightarrow(k_s,s)}\|_2$$

**Quantity 2** — global operator norms with $(Q,n)$ as the column index, computed for **(ov|vo)**:
$$\mathcal{N}_2 = \|\mathbf{L}_{(k_p,p,k_q,q)\leftrightarrow(Q,n)}\|_2\;\|\mathbf{R}_{(k_r,r,k_s,s)\leftrightarrow(Q,n)}\|_2$$
where $\mathbf{L} = [L^{Q=0}|L^{Q=1}|\cdots]$ stacks all $L^Q$ column-wise.

In [ ]:
from scipy.sparse.linalg import LinearOperator, svds as sparse_svds


def build_Q_sector_matrix(e7, Q, nkpts):
    """Materialize M^Q only for SVD-factor quantities that need singular vectors."""
    n1, n2 = e7.shape[3], e7.shape[4]
    n3, n4 = e7.shape[5], e7.shape[6]
    M = np.empty((nkpts * n1 * n2, nkpts * n3 * n4), dtype=np.complex128)
    for k1 in range(nkpts):
        k2 = (k1 - Q) % nkpts
        rs = slice(k1 * n1 * n2, (k1 + 1) * n1 * n2)
        for k3 in range(nkpts):
            cs = slice(k3 * n3 * n4, (k3 + 1) * n3 * n4)
            M[rs, cs] = e7[k1, k2, k3].reshape(n1 * n2, n3 * n4)
    return M


def randomized_svd(M, rank, oversample=16, niter=2, seed=1234):
    """Small dependency-free randomized SVD for the leading singular triplets."""
    m, n = M.shape
    ell = min(min(m, n), rank + oversample)
    rng = np.random.default_rng(seed)
    omega = rng.standard_normal((n, ell)) + 1j * rng.standard_normal((n, ell))
    Y = M @ omega
    for _ in range(niter):
        Y = M @ (M.conj().T @ Y)
    Q, _ = np.linalg.qr(Y, mode="reduced")
    B = Q.conj().T @ M
    Uh, s, Vh = np.linalg.svd(B, full_matrices=False)
    U = Q @ Uh
    return U[:, :rank], s[:rank], Vh[:rank]


def leading_svd(M, method="randomized", max_rank=96, oversample=16, niter=2, seed=1234):
    """Return leading singular triplets sorted from largest to smallest."""
    min_dim = min(M.shape)
    if method == "full" or max_rank is None or max_rank >= min_dim:
        U, s, Vh = np.linalg.svd(M, full_matrices=False)
        return U, s, Vh

    rank = min(max_rank, min_dim - 1)
    if method == "svds":
        U, s, Vh = sparse_svds(M, k=rank, which="LM", return_singular_vectors=True)
        order = np.argsort(s)[::-1]
        return U[:, order], s[order], Vh[order]
    if method == "randomized":
        return randomized_svd(M, rank, oversample=oversample, niter=niter, seed=seed)
    raise ValueError(f"Unknown q1_svd_method={method!r}")


def build_Q_sector_df(e7, nkpts, tol=1e-8, method="randomized", max_rank=96,
                      oversample=16, niter=2, seed=1234):
    """SVD-factorize M^Q for each momentum transfer Q.

    The old notebook used a full dense SVD for every Q. This version defaults to
    a leading randomized SVD, which is much faster for the norm diagnostics. Use
    method="full" for exact factors.
    """
    L_all, R_all = {}, {}
    ranks = {}

    for Q in range(nkpts):
        M = build_Q_sector_matrix(e7, Q, nkpts)
        U, s, Vh = leading_svd(
            M,
            method=method,
            max_rank=max_rank,
            oversample=oversample,
            niter=niter,
            seed=seed + Q,
        )
        keep = s >= tol * max(s[0], 1e-300)
        if not np.any(keep):
            keep[0] = True
        U, s, Vh = U[:, keep], s[keep], Vh[keep]
        sq = np.sqrt(s)
        L_all[Q] = U * sq[None, :]
        R_all[Q] = Vh.conj().T * sq[None, :]
        ranks[Q] = len(s)

    return L_all, R_all, ranks


def block_op_norms_columns(mat, nkpts, na, nb):
    """Column-wise block operator norms for all DF columns at once."""
    rank = mat.shape[1]
    blocks = mat.reshape(nkpts, na, nb, rank).transpose(3, 0, 1, 2)
    svals = np.linalg.svd(blocks.reshape(rank * nkpts, na, nb), compute_uv=False)
    return svals[:, 0].reshape(rank, nkpts).max(axis=1)


def quantity_1(L_all, R_all, nkpts, n1, n2, n3, n4):
    """N1 = sum_{Q,n} ||L^{Q,n}||_blk * ||R^{Q,n}||_blk."""
    total = 0.0
    for Q in sorted(L_all):
        L, R = L_all[Q], R_all[Q]
        l_norms = block_op_norms_columns(L, nkpts, n1, n2)
        r_norms = block_op_norms_columns(R, nkpts, n3, n4)
        total += float(np.dot(l_norms, r_norms))
    return total


def quantity_2(L_all, R_all):
    """N2 = sigma_max(L_full) * sigma_max(R_full)."""
    L_full = np.hstack([L_all[Q] for Q in sorted(L_all)])
    R_full = np.hstack([R_all[Q] for Q in sorted(R_all)])
    return (np.linalg.svd(L_full, compute_uv=False)[0] *
            np.linalg.svd(R_full, compute_uv=False)[0])


In [ ]:
q1_results = {}
print(f"Quantity 1 SVD method = {q1_svd_method}, max_rank = {q1_max_rank}, tol = {q1_tol:g}")
print(f"Quantity 1 blocks     = {q1_blocks}")

for bname in q1_blocks:
    cfg = block_config[bname]
    e7  = get_eri_7d(bname)
    n1, n2 = e7.shape[3], e7.shape[4]
    n3, n4 = e7.shape[5], e7.shape[6]
    print(f"{cfg['label']}  M^Q shape = ({nkpts*n1*n2}, {nkpts*n3*n4}) ...", flush=True)

    L_all, R_all, rank = build_Q_sector_df(
        e7,
        nkpts,
        tol=q1_tol,
        method=q1_svd_method,
        max_rank=q1_max_rank,
        oversample=q1_oversample,
        niter=q1_niter,
        seed=q1_random_seed,
    )
    print(f"  retained rank per Q: {list(rank.values())}")

    q1 = quantity_1(L_all, R_all, nkpts, n1, n2, n3, n4)
    q1_results[bname] = (L_all, R_all, q1)
    print(f"  N1 / Nk = {q1 / nkpts:.6e}")

if "ovvo" in q1_results:
    print()
    print("Quantity 2 for (ov|vo):")
    L_all_ov, R_all_ov, _ = q1_results["ovvo"]
    q2 = quantity_2(L_all_ov, R_all_ov)
    print(f"  N2 / Nk = {q2 / nkpts:.6e}")
else:
    q2 = None
    print()
    print("Quantity 2 skipped because 'ovvo' is not in q1_blocks.")

print()
print("Summary  (all quantities / Nk)")
print(f"{'Block':8s}  {'N1/Nk':>14s}")
print("-" * 28)
for bname, (_, _, q1) in q1_results.items():
    print(f"{block_config[bname]['label']:8s}  {q1 / nkpts:14.6e}")
if q2 is not None:
    print(f"{'(ov|vo)':8s}  N2/Nk = {q2 / nkpts:.6e}")



Quantity 1 SVD method = randomized, max_rank = 96, tol = 1e-08
Quantity 1 blocks     = ['oooo', 'ovvo']
(oo|oo)  M^Q shape = (432, 432) ...
  retained rank per Q: [94, 96, 96, 96, 96, 96, 96, 96, 96, 76, 96, 96, 96, 96, 96, 96, 96, 96, 76, 96, 96, 96, 96, 96, 96, 96, 96]
  N1 / Nk = 2.786537e+00
(ov|vo)  M^Q shape = (2376, 2376) ...
  retained rank per Q: [96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 96]
  N1 / Nk = 3.135127e+00

Quantity 2 for (ov|vo):
  N2 / Nk = 1.702439e+00

Summary  (all quantities / Nk)
Block              N1/Nk
----------------------------
(oo|oo)     2.786537e+00
(ov|vo)     3.135127e+00
(ov|vo)   N2/Nk = 1.702439e+00


### 2-norm of the ERI as a matrix: $(k_p,p,k_q,q)\leftrightarrow(k_s,s,k_r,r)$

The full ERI matrix with rows $(k_p,p,k_q,q)$ and columns $(k_s,s,k_r,r)$ is
**block-diagonal in** $Q = k_p{-}k_q = k_s{-}k_r$, with the $Q$-th block equal to $M^Q$
(column permutations do not affect singular values). Therefore:
$$\|E_{(k_p,p,k_q,q)\leftrightarrow(k_s,s,k_r,r)}\|_2 = \max_Q\,\sigma_{\max}(M^Q)$$

Only the leading singular value per $Q$ is needed. The code below uses a `LinearOperator`, so it applies $M^Q$ and $(M^Q)^\dagger$ directly from the 7D tensor without constructing the dense Q-sector matrix.


In [ ]:
def Q_sector_linear_operator(e7, Q, nkpts):
    """LinearOperator for M^Q without materializing the dense matrix."""
    n1, n2 = e7.shape[3], e7.shape[4]
    n3, n4 = e7.shape[5], e7.shape[6]
    left_dim = nkpts * n1 * n2
    right_dim = nkpts * n3 * n4

    blocks = [e7[k1, (k1 - Q) % nkpts].reshape(nkpts, n1 * n2, n3 * n4)
              for k1 in range(nkpts)]

    def matvec(x):
        x = np.asarray(x).reshape(nkpts, n3 * n4)
        y = np.empty((nkpts, n1 * n2), dtype=np.complex128)
        for k1, A in enumerate(blocks):
            y[k1] = np.einsum("kab,kb->a", A, x, optimize=True)
        return y.ravel()

    def rmatvec(x):
        x = np.asarray(x).reshape(nkpts, n1 * n2)
        y = np.zeros((nkpts, n3 * n4), dtype=np.complex128)
        for k1, A in enumerate(blocks):
            y += np.einsum("kab,a->kb", A.conj(), x[k1], optimize=True)
        return y.ravel()

    return LinearOperator((left_dim, right_dim), matvec=matvec, rmatvec=rmatvec, dtype=np.complex128)


def eri_2norm(e7, nkpts, tol=1e-6):
    """||E||_2 = max_Q sigma_max(M^Q), using matrix-free rank-1 SVD."""
    sigma_max = 0.0
    for Q in range(nkpts):
        op = Q_sector_linear_operator(e7, Q, nkpts)
        s1 = float(sparse_svds(op, k=1, tol=tol, return_singular_vectors=False)[0])
        sigma_max = max(sigma_max, s1)
    return sigma_max


print(f"{'Block':8s}  {'||E||_2 / Nk':>14s}  (matrix-free over Q sectors)")
print("-" * 40)
for bname in norm2_blocks:
    norm2 = eri_2norm(get_eri_7d(bname), nkpts, tol=norm2_tol)
    print(f"{block_config[bname]['label']:8s}  {norm2 / nkpts:14.6e}")



Block       ||E||_2 / Nk  (matrix-free over Q sectors)
----------------------------------------
(oo|oo)     1.939395e+00
(ov|vo)     4.054205e-01


## Optional: Save Results

In [ ]:
if save_npz:
    out_npz.parent.mkdir(parents=True, exist_ok=True)
    payload = {}
    for bname, e7 in eri_7d.items():
        payload[f"{bname}_7d"] = np.asarray(e7)
    for bname, es in eri_spc.items():
        payload[f"{bname}_spc"] = es
    np.savez_compressed(out_npz, **payload)
    print(f"Saved {len(payload)} arrays to {out_npz}")
else:
    print("Set `save_npz = True` to save built results to disk.")


Set `save_npz = True` to save built results to disk.
